# 16 — API Design, Versioning, and Contracts

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/16', {1: 'b3c6e139ee93ebba', 2: '692ee2bc2abdfad8', 3: 'c5fef6c2e1edb91b', 4: 'da5a9597dfece5e8', 5: '18f6c671a1dbb32c', 6: '8538872766d0bd7c', 7: 'bd60fd85b111ec20', 8: 'b491130784e25a1f', 9: 'f6f96e40d8222903'}, practice='advanced_04_production_systems')

## 1. Stable contracts

- A published response shape is a contract: never remove or rename fields in a version clients use.
- Add new fields in a new version (or as optional fields); keep v1 output exactly the same.
- Reject unknown versions with a clear error instead of guessing.

In [ ]:
def serialize(user, version=1):
    if version not in (1, 2):
        raise ValueError(f"unsupported version {version}")
    body = {"id": user["id"], "name": user["name"]}
    if version == 2:
        body["email"] = user.get("email")
    return body

u = {"id": 1, "name": "Ana", "password_hash": "x"}
print(serialize(u), serialize(u, 2))

**Exercise 1.** v2 adds an optional `email`. What does v2 give for a user without one?

In [ ]:
user = {"id": 3, "name": "Bo"}
body = {"id": user["id"], "name": user["name"], "email": user.get("email")}
predicted = ___
check(1, predicted)

**Exercise 2.** Only copy public fields, never internal ones like `password_hash`.

In [ ]:
user = {"id": 1, "name": "Ana", "password_hash": "x"}
public = {k: user[k] for k in ___}
check(2, public)

**Exercise 3.** Which versions are supported? Keep the rest out.

In [ ]:
requested = [1, 2, 3, 0]
supported = [v for v in requested if v in ___]
check(3, supported)

## 2. Error envelopes

- Return errors in one consistent shape: `{"error": {"code", "message", "status"}}`.
- For 5xx errors hide internal details: the message becomes `"Internal server error"`.
- Codes are for programs (`"not_found"`); messages are for people.

In [ ]:
def envelope(code, message, status):
    safe = "Internal server error" if status >= 500 else message
    return {"error": {"code": code, "message": safe, "status": status}}

print(envelope("not_found", "No such order", 404))
print(envelope("db_down", "psycopg: connection refused at 10.0.0.5", 503))

**Exercise 4.** Hide the details of a 500 error.

In [ ]:
status, message = 500, "KeyError in orders.py line 88"
safe = "Internal server error" if status >= ___ else message
check(4, safe)

**Exercise 5.** Build the envelope for a 404.

In [ ]:
body = {"error": {"code": "not_found", "message": "No such user", "status": ___}}
check(5, body["error"])

## 3. Cursors and semantic versions

- Cursor pagination: each page returns `cursor` for the next one; stop when it's `None`. Fail if a cursor repeats (the server is looping).
- Semantic versions `MAJOR.MINOR.PATCH`: major breaks compatibility, minor adds features, patch fixes bugs.
- A client works with the same major version and an equal or newer minor.

In [ ]:
def parse_version(text):
    parts = text.removeprefix("v").split(".")
    if len(parts) != 3 or not all(p.isdigit() for p in parts):
        raise ValueError(text)
    return tuple(int(p) for p in parts)

print(parse_version("v2.10.1") > parse_version("2.9.9"))
c, s = parse_version("1.2.0"), parse_version("1.4.3")
print(c[0] == s[0] and s[1] >= c[1])

**Exercise 6.** Parse `"v3.1.4"` into a tuple of ints.

In [ ]:
parts = "v3.1.4".removeprefix("" + ___).split(".")
check(6, tuple(int(p) for p in parts))

**Exercise 7.** Compare versions as tuples, not strings: is `2.10.0` newer than `2.9.0`?

In [ ]:
predicted = ___
check(7, predicted)

**Exercise 8.** Client 2.3.0 with servers 2.5.1, 3.0.0 and 2.1.0. Which are compatible?

In [ ]:
client = (2, 3, 0)
servers = [(2, 5, 1), (3, 0, 0), (2, 1, 0)]
ok = [s for s in servers if s[0] == client[0] and s[1] >= ___]
check(8, ok)

**Exercise 9.** Follow cursors until `None`.

In [ ]:
pages = {None: {"items": [1], "cursor": "c2"}, "c2": {"items": [2, 3], "cursor": None}}
items, cursor = [], None
for _ in range(10):   # a safety limit
    page = pages[cursor]
    items.extend(page["items"])
    cursor = page["cursor"]
    if cursor is ___:
        break
check(9, items)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `serialize_user`

v1 and v2 user bodies; reject other versions.

In [ ]:
def serialize_user(user: dict, version: int = 1) -> dict:
    """v1 -> {'id', 'name'}; v2 adds optional 'email' (None if absent). Other versions: ValueError.

    v1 output must never change: old clients depend on it.
    """
    if ___:
        raise ValueError(f"unsupported version {version}")
    body = {"id": user["id"], "name": user["name"]}
    if ___:
        body["email"] = user.get("email")
    return body

In [ ]:
check('serialize_user')

### 2. `error_envelope`

A consistent error body that hides 5xx details.

In [ ]:
def error_envelope(code: str, message: str, status: int) -> dict:
    """Build {'error': {'code', 'message', 'status'}}. For status >= 500 hide internal details:
    message becomes 'Internal server error'."""
    safe_message = "Internal server error" if status >= 500 else message
    return ___

In [ ]:
check('error_envelope')

### 3. `iterate_cursor`

Follow cursors; raise `RuntimeError` if one repeats.

In [ ]:
def iterate_cursor(fetch: Callable[[str | None], dict]) -> list:
    """Follow {'items': [...], 'cursor': next or None} starting from fetch(None).

    Raise RuntimeError if a cursor repeats (the server is looping).
    """
    items, seen, cursor = [], set(), None
    while ___:
        page = fetch(cursor)
        items.extend(page["items"])
        cursor = page.get("cursor")
        if ___:
            return items
        if ___:
            raise RuntimeError(f"cursor repeated: {cursor}")
        seen.add(cursor)

In [ ]:
check('iterate_cursor')

### 4. `parse_version`

Parse `MAJOR.MINOR.PATCH` (optional `v`) into ints.

In [ ]:
def parse_version(text: str) -> tuple[int, int, int]:
    """Parse 'MAJOR.MINOR.PATCH' (optional leading 'v') into ints; ValueError otherwise."""
    parts = text.removeprefix("v").split(".")
    if ___:
        raise ValueError(f"not a semantic version: {text!r}")
    return ___

In [ ]:
check('parse_version')

### Independent build — `is_compatible`

Given two major.minor.patch version strings (ignore patch), return True exactly when major versions match and the server minor version is at least the client minor version. Compare numeric components.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def is_compatible(client: str, server: str) -> bool:
    raise NotImplementedError

In [ ]:
check('is_compatible')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Adding an optional field to an API response is usually…

   - **a)** Backward compatible
   - **b)** Forbidden
   - **c)** A breaking change

**2.** Cursor pagination should protect against…

   - **a)** Small pages
   - **b)** JSON
   - **c)** Repeated cursors (infinite loops)

**3.** A stable error response includes…

   - **a)** A machine-readable code and a safe message
   - **b)** A stack trace
   - **c)** Database credentials

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/16', {1: '37f1b73c95ce', 2: '8d26c0292075', 3: '758e2ed85f7f'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [17 — Transactions, Constraints, and Exactly-Once Semantics](17_sqlite_transactions_integrity.ipynb)